# Step 3 — Neural Network (LSTM)

In this step, I build and train a neural network using the same cleaned dataset from Step 1.  
Unlike the machine‑learning models in Step 2, this model learns from sequences of words using an Embedding layer and an LSTM layer.

My goals in this step are:
- Convert text into sequences of integers  
- Pad all sequences to the same length  
- Build an LSTM‑based neural network  
- Train the model for multiple epochs  
- Evaluate the model using accuracy, precision, recall, and F1‑score  
- Compare the neural network with the ML models from Step 2  

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score

from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

## 1. Loading the Cleaned Dataset

I load the same cleaned dataset I used in Step 2.  
This ensures that both the ML models and the neural network are trained on identical data.

In [ ]:
df = pd.read_csv("cleaned_fake_news.csv")
df.head()

## 2. Train/Test Split

I use the same split strategy as Step 2:
- 80% training  
- 20% testing  
- stratify=y to preserve class balance  

This ensures a fair comparison between ML models and the neural network.

In [ ]:
X = df["clean_text"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

len(X_train), len(X_test)

## 3. Tokenization

Neural networks cannot work with raw text, so I convert each word into an integer using Keras Tokenizer.

Steps:
- Build a vocabulary of the most frequent words  
- Convert each text into a sequence of integers  
- Unknown words are replaced with `<OOV>`  

In [ ]:
vocab_size = 10000  

tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

## 4. Padding Sequences

Neural networks require all sequences to have the same length.  
I pad shorter sequences with zeros at the end.

I choose a maximum length of 200 words per article.

In [ ]:
max_len = 200

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

X_train_pad.shape, X_test_pad.shape

## 5. Building the LSTM Model

I build a neural network with the following architecture:

- **Embedding layer**: learns word representations  
- **LSTM layer (64 units)**: captures sequence patterns  
- **Dropout layers**: reduce overfitting  
- **Dense layers**: final classification  

The output layer uses a sigmoid activation because this is a binary classification task.

In [ ]:
model = Sequential([
    Embedding(input_dim=vocab_size, output_dim=64, input_length=max_len),
    LSTM(64, return_sequences=False),
    Dropout(0.3),
    Dense(32, activation='relu'),
    Dropout(0.2),
    Dense(1, activation='sigmoid')
])

model.compile(
    loss='binary_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

## 6. Training the Model

I train the model for 10 epochs as required in the capstone instructions.  
I also use a validation split to monitor overfitting.

In [ ]:
history = model.fit(
    X_train_pad, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.2
)

## 7. Training Curves

I plot the training and validation accuracy/loss to visualize the model’s learning progress.

In [ ]:
plt.figure(figsize=(14,5))

# Accuracy
plt.subplot(1,2,1)
plt.plot(history.history['accuracy'], label='Train Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title("Accuracy Over Epochs")
plt.legend()

# Loss
plt.subplot(1,2,2)
plt.plot(history.history['loss'], label='Train Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title("Loss Over Epochs")
plt.legend()

plt.show()

## 8. Evaluation on Test Set

I evaluate the neural network using the same metrics as Step 2:
- Accuracy  
- Precision  
- Recall  
- F1-score  

In [ ]:
y_pred_nn = (model.predict(X_test_pad) > 0.5).astype(int)

print(classification_report(y_test, y_pred_nn))

## 9. Comparison with Step 2

I compare the neural network results with the ML models from Step 2.

In most text‑classification tasks:
- LSTM models capture deeper patterns in language  
- Logistic Regression performs strongly with TF‑IDF  
- Random Forest and KNN usually perform worse on sparse text data  

My final comparison will be included in the written report.